<a href="https://colab.research.google.com/github/GoudoMahan/AI-agent-practice/blob/main/HM_3_1_nuscene_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 作业3-任务1：认识自动驾驶感知数据集 nuscene
---

姓名：胡豪达

学号：523021910471

---

本 Task 占比本作业评分的 50%，分为两个部分：

| 部分 | 内容 | 分值 |
|------|------|------|
| Part 1 |  了解各个传感器采集的数据 | 50 |
| Part 2 | 了解不同的感知任务对应的数据标注 | 50 |

**⚠️ 重要提示：**
- 选择 **CPU 运行时**（`运行时 → 更改运行时类型 → CPU`）
- <mark> 👉  高亮块是你需要完成或者修改的内容提示，你需要根据任务要求修改对应地方下方的代码块 </mark>，其余代码块请按顺序运行

---
## 🔧 环境安装

下载并解压数据集，大约需要 2 分钟

In [ ]:
!mkdir -p /data/sets/nuscenes  # Make the directory to store the nuScenes dataset in.

!wget https://www.nuscenes.org/data/v1.0-mini.tgz  # Download the nuScenes mini split.

!tar -xf v1.0-mini.tgz -C /data/sets/nuscenes  # Uncompress the nuScenes mini split.

若出现红色 ERROR 请继续不影响本作业

In [ ]:
# Devkit base deps (numpy<2, etc.): https://github.com/nutonomy/nuscenes-devkit/blob/master/setup/requirements/requirements_base.txt
%pip install -q -r https://raw.githubusercontent.com/nutonomy/nuscenes-devkit/master/setup/requirements/requirements_base.txt
%pip install -q nuscenes-devkit
%pip install -q imageio imageio-ffmpeg
%pip install -q plotly

![2026-04-21-15-18-17](https://i.ibb.co/ZRgprnxs/2026-04-21-15-18-17.png)

> ⚠️：请点击 重启会话，后运行下面的代码

In [ ]:
%matplotlib inline
from nuscenes.nuscenes import NuScenes

nusc = NuScenes(version='v1.0-mini', dataroot='/data/sets/nuscenes', verbose=False)

## 1. 认识传感器

nuScenes 数据集（发音为/nuːsiːnz/）是由 Motional 公司（前身为 nuTonomy）团队开发的、用于自动驾驶领域的公共大型数据集。nuScenes 数据包含了波士顿和新加坡约 15 小时的驾驶数据。为了便于处理常见的计算机视觉任务，如物体检测和跟踪，nuScenes 团队在整个数据集上以 2Hz 的频率，为 23 个物体类别标注了精确的 3D 边界框和物体级别的属性，如可见性、活跃程度和姿态。

2019 年 3 月，nuScenes 团队发布了包含全部 1,000 个场景的 nuScenes 完整数据集。该数据集包含约 140 万张相机图像、39 万次激光雷达扫描数据、140 万次雷达扫描数据，以及 4 万个关键帧中的 140 万个物体边界框。作为首个大规模数据集，nuScenes 提供了自动驾驶车辆全部传感器设备的数据（6 台摄像头、1 个激光雷达、5 个雷达、GPS、惯性测量单元）。

下图是各个传感器的具体位置。

<a href="https://ibb.co/hxRFXz9T"><img src="https://i.ibb.co/rKG2br0X/nuscene-1.png" alt="nuscene-1" border="0"></a>

### 1.1 摄像头 CAMERA

数据采集系统包含 6 台相机，分辨率均为 1600×1200，采集频率为 12 Hz，视场合计覆盖 360°。图像先解码为 BGR，再编码为 JPEG。在 devkit 里通常记为 `CAM_FRONT_LEFT`、`CAM_FRONT`、`CAM_FRONT_RIGHT`、`CAM_BACK_LEFT`、`CAM_BACK`、`CAM_BACK_RIGHT`。

<a href="https://imgbb.com/"><img src="https://i.ibb.co/tpWWZ4H3/nuscene-2.jpg" alt="nuscene 2" border="0"></a>


你可以尝试更改随机种子，以获取不同场景下的可视化。

In [ ]:
import random

rng = random.Random(20010101)    # <-- 你可以修改这里的随机种子
scene = rng.choice(nusc.scene)
first_sample = nusc.get("sample", scene["first_sample_token"])

运行下面的代码，我们可以得到场景首帧的六个视角的图像。

In [ ]:

from typing import Tuple

import matplotlib.pyplot as plt
from PIL import Image

CAM_CHANNELS: Tuple[str, ...] = (
    "CAM_FRONT_LEFT",
    "CAM_FRONT",
    "CAM_FRONT_RIGHT",
    "CAM_BACK_LEFT",
    "CAM_BACK",
    "CAM_BACK_RIGHT",
)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, ch in zip(axes.flatten(), CAM_CHANNELS):
    sd_token = first_sample["data"][ch]
    impath, _, _ = nusc.get_sample_data(sd_token)
    ax.imshow(Image.open(impath))
    ax.set_title(ch, fontsize=10)
    ax.axis("off")

fig.suptitle(f"Six cameras at one keyframe | sample_token={first_sample['token']}", fontsize=12)
plt.tight_layout()
plt.show()

运行下面的代码，我们可以得到上方获取的帧对应场景中 20s 的六个视角的视频，耗时约需 2 分钟。

In [ ]:
from typing import Any, Dict, List, Tuple

import numpy as np
from IPython.display import Video, display
from PIL import Image

import imageio.v2 as imageio

# 12 Hz playback, up to 20 s (240 frames). Walk each camera's sample_data chain (~12 Hz),
# same time-sync idea as NuScenesExplorer.render_scene in nuscenes-devkit.

FPS: int = 12
DURATION_SEC: int = 20
MAX_FRAMES: int = FPS * DURATION_SEC

CAM_CHANNELS: Tuple[str, ...] = (
    "CAM_FRONT_LEFT",
    "CAM_FRONT",
    "CAM_FRONT_RIGHT",
    "CAM_BACK_LEFT",
    "CAM_BACK",
    "CAM_BACK_RIGHT",
)
HORIZONTAL_FLIP: frozenset = frozenset({"CAM_BACK_LEFT", "CAM_BACK", "CAM_BACK_RIGHT"})

LAYOUT: Dict[str, Tuple[int, int]] = {
    "CAM_FRONT_LEFT": (0, 0),
    "CAM_FRONT": (1, 0),
    "CAM_FRONT_RIGHT": (2, 0),
    "CAM_BACK_LEFT": (0, 1),
    "CAM_BACK": (1, 1),
    "CAM_BACK_RIGHT": (2, 1),
}


def build_frame_grid_from_sd_tokens(sd_tokens: Dict[str, str], tile_w: int, tile_h: int) -> np.ndarray:
    canvas = np.zeros((2 * tile_h, 3 * tile_w, 3), dtype=np.uint8)
    for ch in CAM_CHANNELS:
        col, row = LAYOUT[ch]
        x0, y0 = col * tile_w, row * tile_h
        impath, _, _ = nusc.get_sample_data(sd_tokens[ch])
        im = np.asarray(Image.open(impath).convert("RGB"), dtype=np.uint8)
        im = np.asarray(Image.fromarray(im).resize((tile_w, tile_h)))
        if ch in HORIZONTAL_FLIP:
            im = im[:, ::-1, :]
        canvas[y0 : y0 + tile_h, x0 : x0 + tile_w] = im
    return canvas

first_sample = nusc.get("sample", scene["first_sample_token"])
last_sample = nusc.get("sample", scene["last_sample_token"])

t0 = int(first_sample["timestamp"])
t_end = min(t0 + int(DURATION_SEC * 1e6), int(last_sample["timestamp"]))
dt_us = int(round(1e6 / FPS))

current_sd: Dict[str, Dict[str, Any]] = {
    ch: nusc.get("sample_data", first_sample["data"][ch]) for ch in CAM_CHANNELS
}

tile_w, tile_h = 256, 144
frames: List[np.ndarray] = []
for i in range(MAX_FRAMES):
    t = t0 + i * dt_us
    if t > t_end:
        break
    for ch in CAM_CHANNELS:
        sd = current_sd[ch]
        while sd["timestamp"] < t and sd["next"]:
            sd = nusc.get("sample_data", sd["next"])
            current_sd[ch] = sd
    sd_tokens = {ch: current_sd[ch]["token"] for ch in CAM_CHANNELS}
    frames.append(build_frame_grid_from_sd_tokens(sd_tokens, tile_w, tile_h))

out_path = "/tmp/nuscenes_sixcam_scene.mp4"
imageio.mimsave(out_path, frames, fps=FPS)

actual_sec = len(frames) / FPS
print(
    f"scene name={scene['name']} | frames={len(frames)} | {FPS} fps | "
    f"~{actual_sec:.2f}s playback | saved {out_path}"
)
display(Video(out_path, embed=True))


### 1.2 激光雷达 LIDAR

nuScenes 在车顶中央部署 **1 台 Velodyne HDL-32E** 旋转式激光雷达（通道名 `LIDAR_TOP`）。官方标称最大测距约 **80–100 m**，在实际道路回波与车体遮挡等因素下，**有效探测距离常取约 70 m**；距离测量精度典型为 **±2 cm**。传感器为 **32 线**，水平方向 **360°** 一圈扫描，垂直视场约 **+10°～−30°**；水平角分辨率约 **1080** 点/圈（实际每圈点数可有约 **±10** 点的波动）。HDL-32E 转速可在约 **5–20 Hz** 范围内配置；**在 nuScenes 中 LiDAR 以 20 Hz 采集**（每秒约 **20** 帧完整扫描）。**关键帧及 3D 框标注为 2 Hz**，因此在两条关键帧之间通常还存在多点云扫描仅用于感知融合、不一定对应独立标注帧。**点云生成速率约每秒 139 万点**。

一般来说，点云包含多个通道信息：
- x, y, z ：反射点的三维空间位置
- intensity： 反射强度，和物体的材质属性有关
- ring：线束索引，该点所属的激光线束编号，例如 nuScenes 使用 32 线雷达，该值可以取 0 - 31。它存的是每条回波属于哪一束光，是点的一个离散标号，用来区分 32 个不同俯仰角的发射通道。



运行下方代码可加载某一帧 `LIDAR_TOP` 点云，并在 **可交互三维视图**（鼠标拖拽旋转、滚轮缩放）中查看

如果下述代码在 colab 网页版可视化遇到问题，可以在 IDE ( VSCode, Cursor 等) 中打开本文件，安装 colab 插件后，选择 kernel 为 colab 重新运行。

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio

import os
import random

import numpy as np
from nuscenes.utils.data_classes import LidarPointCloud


MAX_POINTS: int = 80000


def subsample_points(xyz: np.ndarray, intensities: np.ndarray | None, rng: np.random.Generator, max_n: int) -> tuple[np.ndarray, np.ndarray | None]:
    """Randomly subsample LiDAR points for faster Plotly rendering."""
    n = xyz.shape[0]
    if n <= max_n:
        return xyz, intensities
    idx = rng.choice(n, size=max_n, replace=False)
    xyz_s = xyz[idx]
    int_s = intensities[idx] if intensities is not None else None
    return xyz_s, int_s


def configure_plotly_notebook_renderer() -> None:
    """
    配置链式渲染器，实现跨前端兼容。
    生成的输出将包含多种格式，VS Code 会识别 'vscode' 格式，
    而网页端 Colab 会识别 'colab' 所需的注入脚本。
    """
    pio.renderers.default = "vscode+colab+notebook_connected"

RNG_SEED: int = 123
rng_py = random.Random(RNG_SEED)
rng_np = np.random.default_rng(RNG_SEED)

# 沿用前一个 cell 中选取的场景的数据
lidar_sd_token = first_sample["data"]["LIDAR_TOP"]
sd = nusc.get("sample_data", lidar_sd_token)
pcl_path = os.path.join(nusc.dataroot, sd["filename"])
print(f"从文件 {pcl_path} 中加载点云数据...")

pc = LidarPointCloud.from_file(pcl_path)
pts = pc.points  # shape (C, N); first rows: x,y,z,intensity,...
xyz = pts[:3].T.astype(np.float64)
intensity = pts[3].astype(np.float64) if pts.shape[0] > 3 else None
print(f"成功加载点云，数量: {pts.shape[-1]}, 通道：{pts.shape[0]}")

xyz_vis, int_vis = subsample_points(xyz, intensity, rng_np, MAX_POINTS)

color = int_vis if int_vis is not None else xyz_vis[:, 2]
cmin = float(np.percentile(color, 2))
cmax = float(np.percentile(color, 98))

configure_plotly_notebook_renderer()
fig = go.Figure(
    data=[
        go.Scatter3d(
            x=xyz_vis[:, 0],
            y=xyz_vis[:, 1],
            z=xyz_vis[:, 2],
            mode="markers",
            marker=dict(
                size=1.5,
                color=color,
                colorscale="Viridis",
                opacity=0.85,
                cmin=cmin,
                cmax=cmax,
                colorbar=dict(title="Intensity" if int_vis is not None else "Height z (m)"),
            ),
        )
    ]
)
fig.update_layout(
    title=f"LIDAR_TOP | scene sample token={first_sample['token'][:8]}… | file={sd['filename']}",
    scene=dict(
        xaxis_title="x (m)",
        yaxis_title="y (m)",
        zaxis_title="z (m)",
        aspectmode="data",
        bgcolor="rgb(20,24,28)",
    ),
    paper_bgcolor="rgb(24,28,32)",
    font=dict(color="#e8e8e8"),
    margin=dict(l=0, r=0, t=40, b=0),
    height=640,
)
fig.show()

此外，我们还可以依据**针孔相机模型**，利用**相机内参**（焦距、主点等，组成 **3×3** 内参矩阵 $K$ ）以及**雷达到相机的外参**（旋转矩阵 $R$ 与平移向量 $t$），先将 LiDAR 坐标系下的三维点变换到相机坐标系，再投影到像素平面得到 $(u, v)$，从而把点云叠画在相机图像上，便于多传感器融合与结果查验；只有落在相机前方视锥内、且外参标定准确、时间同步可靠时，投影才与真实场景对齐。

投影后你看到的一条条结构，主要来自 LiDAR 的几何采样方式。同一俯仰角上，随着水平角扫描排成一条空间曲线，投到图像上就变成一条（或一段）连续的线/弧。

In [ ]:
nusc.render_pointcloud_in_image(first_sample['token'], pointsensor_channel='LIDAR_TOP')

为了实现激光雷达与摄像头之间的良好跨模态数据对齐，当最上方的激光雷达扫描到摄像头视野中心时，摄像头会开始曝光。图像的时间戳即为曝光触发时间；而激光雷达扫描的时间戳则是指当前激光雷达扫描帧完成完整旋转的时刻。由于摄像头的曝光时间几乎为瞬时过程，该方法通常能实现良好的数据对齐效果。需要说明的是，摄像头的刷新率为 12Hz，而激光雷达为 20Hz。12 次摄像头曝光会尽可能均匀地分布在 20 次激光雷达扫描中，因此并非所有的激光雷达扫描都对应有摄像头图像。将摄像头刷新率降至 12Hz 有助于降低感知系统的计算量、带宽需求和存储成本。

<mark> 👉 通过 ai 助手的帮助，请绘制出把上述点云 （ `pc = LidarPointCloud.from_file(pcl_path).points` ）体素化后可视化的图片。并讨论，有哪些处理预处理点云数据的方法？为什么我们需要他们？</mark>

> 点云体素化（Voxelization）是指将三维空间中无序、稀疏的散乱点集，按空间几何划分并映射到规则的、具有固定尺寸的三维离散网格（体素）中的数据处理过程。 其核心作用是将非结构化数据转化为结构化表达，在大幅降低数据量与计算复杂度的同时，使得标准的三维卷积神经网络（3D CNN）操作以及机器人的高效空间索引（如碰撞检测、三维建图）成为可能。


In [ ]:
# 在这里实现你的代码

### 1.3 RADAR

车载毫米波雷达向周围环境发射电磁波并接收回波，用来估计目标**离车多远**、以及目标**沿雷达视线方向的径向速度**（靠近或远离）。在雨雾等能见度下降时，雷达往往比可见光相机更稳定，但仍会受到多径反射、金属护栏旁“鬼影”等因素影响。这类雷达常用调频连续波发射频率随时间线性变化的连续波，从发射与回波的频率差（拍频）可换算径向距离；同时利用回波相对载波的多普勒频移得到径向速度。

nuScenes 采集车安装 5 台 远距毫米波雷达，围绕车身布置以实现近似 360° 感知覆盖；在 devkit 里通常记为 `RADAR_FRONT`、`RADAR_FRONT_LEFT`、`RADAR_FRONT_RIGHT`、`RADAR_BACK_LEFT`、`RADAR_BACK_RIGHT`。最大探测距离约 250 m、采集频率约 13 Hz、速度测量精度约 ±0.1 km/h。

**与 LiDAR / 相机相比 **：单帧雷达点非常**稀疏**（典型目标框内在 10 m 距离上大约数十个回波、更远距离更少），**横向位置**精度通常不如 LiDAR；但雷达能在更远距离上提供**径向速度**信息，与 LiDAR 的几何稠密、相机的语义纹理形成互补。

运行下方代码可加载某一帧 `RADAR` 点云，并在 **可交互三维视图**（鼠标拖拽旋转、滚轮缩放）中查看

In [ ]:
from __future__ import annotations

import os

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from nuscenes.utils.data_classes import RadarPointCloud
from pyquaternion import Quaternion

RADAR_CHANNELS: tuple[str, ...] = (
    "RADAR_FRONT",
    "RADAR_FRONT_LEFT",
    "RADAR_FRONT_RIGHT",
    "RADAR_BACK_LEFT",
    "RADAR_BACK_RIGHT",
)

# Per-mounting-direction visibility at first render (False = hidden until you enable it in the legend).
RADAR_VISIBILITY: dict[str, bool] = {c: True for c in RADAR_CHANNELS}

RADAR_MARKER_COLORS: tuple[str, ...] = (
    "#ff6b6b",
    "#4ecdc4",
    "#ffe66d",
    "#95e1d3",
    "#aa6ff0",
)


def configure_plotly_notebook_renderer() -> None:
    """
    配置链式渲染器，实现跨前端兼容。
    生成的输出将包含多种格式，VS Code 会识别 'vscode' 格式，
    而网页端 Colab 会识别 'colab' 所需的注入脚本。
    """
    pio.renderers.default = "vscode+colab+notebook_connected"


def sensor_xyz_to_ego(points_sensor: np.ndarray, cs_record: dict) -> np.ndarray:
    """Map (3, N) sensor-frame points to ego vehicle frame."""
    rot = Quaternion(cs_record["rotation"]).rotation_matrix.astype(np.float64)
    trans = np.asarray(cs_record["translation"], dtype=np.float64).reshape(3, 1)
    return rot @ points_sensor + trans


def load_radar_points_ego(nusc: object, sample_data_token: str) -> tuple[np.ndarray, str]:
    sd = nusc.get("sample_data", sample_data_token)
    path = os.path.join(nusc.dataroot, sd["filename"])
    RadarPointCloud.disable_filters()
    try:
        pc = RadarPointCloud.from_file(path)
    finally:
        RadarPointCloud.default_filters()
    pts = pc.points
    xyz_s = pts[:3].astype(np.float64)
    cs = nusc.get("calibrated_sensor", sd["calibrated_sensor_token"])
    xyz_ego = sensor_xyz_to_ego(xyz_s, cs)
    return xyz_ego.T, sd["filename"]


configure_plotly_notebook_renderer()

# 与上方随机场景一致：使用已载入的 nusc 与 scene
traces: list[go.Scatter3d] = []
for i, ch in enumerate(RADAR_CHANNELS):
    sd_token = first_sample["data"][ch]
    xyz, _fname = load_radar_points_ego(nusc, sd_token)
    traces.append(
        go.Scatter3d(
            x=xyz[:, 0],
            y=xyz[:, 1],
            z=xyz[:, 2],
            mode="markers",
            name=ch.replace("RADAR_", ""),
            legendgroup=ch,
            visible=RADAR_VISIBILITY.get(ch, True),
            marker=dict(size=3.5, color=RADAR_MARKER_COLORS[i], opacity=0.92),
        )
    )

fig = go.Figure(data=traces)
fig.update_layout(
    title=(
        f"Five radars (ego frame) | sample={first_sample['token'][:8]}… | "
        "legend: click an entry to show/hide that radar"
    ),
    scene=dict(
        xaxis_title="x forward (m)",
        yaxis_title="y left (m)",
        zaxis_title="z up (m)",
        aspectmode="data",
        bgcolor="rgb(20,24,28)",
    ),
    paper_bgcolor="rgb(24,28,32)",
    font=dict(color="#e8e8e8"),
    margin=dict(l=0, r=0, t=48, b=0),
    height=640,
    legend=dict(title="Direction (toggle)", bgcolor="rgba(0,0,0,0.35)"),
    uirevision="radar5",
)
fig.show()

相似地，我们可以把 radar 叠画在相机图像上。

In [ ]:
nusc.render_pointcloud_in_image(first_sample['token'], pointsensor_channel='RADAR_FRONT')

### 1.4 惯性测量单元 IMU 与全球定位

采集车搭载 1 套 GPS、MEMS 型 IMU 与 AHRS 组合单元。其中，RTK 平面位置精度为 20 mm、航向精度为约 0.2°、滚转与俯仰精度为各约 **0.1°、时间更新率 1000 Hz。

在 devkit 的常规分割里：`ego_pose` 表示融合定位后的车辆位姿

运行下方代码：仅使用 **`ego_pose` 链**（与 `LIDAR_TOP` 关键帧同步）绘制 **地面 (x, y) 轨迹**，并对相邻位姿做差分得到**估计车速—时间**曲线。

In [ ]:
from __future__ import annotations

from typing import List

import matplotlib.pyplot as plt
import numpy as np

# 与上方随机场景一致：使用已载入的 nusc 与 scene
SD_CHANNEL: str = "LIDAR_TOP"


def load_scene_ego_path(nusc: object, scene: dict) -> tuple[np.ndarray, np.ndarray, List[str]]:
    """Per keyframe: global translation and timestamp from the ego_pose linked to LIDAR_TOP."""
    ts: List[int] = []
    xyz: List[np.ndarray] = []
    s_tokens: List[str] = []
    st = scene["first_sample_token"]
    while st:
        sm = nusc.get("sample", st)
        sd = nusc.get("sample_data", sm["data"][SD_CHANNEL])
        ego = nusc.get("ego_pose", sd["ego_pose_token"])
        ts.append(int(ego["timestamp"]))
        xyz.append(np.asarray(ego["translation"], dtype=np.float64))
        s_tokens.append(st)
        st = sm["next"]
    if not ts:
        return np.zeros((0, 3), np.float64), np.array([], np.int64), []
    return np.vstack(xyz), np.array(ts, dtype=np.int64), s_tokens


def speed_from_consecutive_poses(translation: np.ndarray, t_us: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Return relative mid-times (s) and speed (km/h) between keyframes."""
    n = int(translation.shape[0])
    if n < 2:
        return np.array([]), np.array([])
    t_sec = t_us.astype(np.float64) * 1.0e-6
    t0 = float(t_sec[0])
    t_rel = t_sec - t0
    dt = np.diff(t_sec)
    dp = np.diff(translation, axis=0)
    dist = np.linalg.norm(dp, axis=1)
    v_m_s = dist / np.maximum(dt, 1e-9)
    t_mid = 0.5 * (t_rel[:-1] + t_rel[1:])
    v_kmh = v_m_s * 3.6
    return t_mid, v_kmh


p_w, t_us, _stok = load_scene_ego_path(nusc, scene)
t_rel_mid, v_kmh = speed_from_consecutive_poses(p_w, t_us)

fig, (ax_t, ax_v) = plt.subplots(1, 2, figsize=(16, 5.0))
if p_w.shape[0] == 0:
    raise RuntimeError("No ego samples in this scene; check nusc/mini and scene choice.")
ax_t.plot(p_w[:, 0], p_w[:, 1], color="#1f77b4", linewidth=1.2, alpha=0.9, label="ego path")
ax_t.plot(p_w[0, 0], p_w[0, 1], "o", color="#2ca02c", markersize=8, label="start", zorder=3)
ax_t.plot(p_w[-1, 0], p_w[-1, 1], "s", color="#d62728", markersize=7, label="end", zorder=3)
ax_t.set_aspect("equal", adjustable="box")
ax_t.set_xlabel("Global x (m)")
ax_t.set_ylabel("Global y (m)")
ax_t.set_title("Ground trajectory from ego_pose (2 Hz keyframes, nuScenes global frame)")
ax_t.legend()
ax_t.grid(True, alpha=0.3)

if t_rel_mid.size:
    ax_v.plot(t_rel_mid, v_kmh, color="#ff7f0e", linewidth=1.3, label="finite-difference |v|")
    ax_v.set_ylabel("Speed (km/h)")
    ax_v.set_title("Per-interval speed (from pose deltas, not raw IMU or wheel ticks)")
else:
    ax_v.set_title("Need at least 2 keyframes to plot speed.")
ax_v.set_xlabel("Time in scene (s)")
ax_v.grid(True, alpha=0.3)
ax_v.legend()

fig.suptitle(f"IMU / localization intuition | scene={scene['name']} | {p_w.shape[0]} keyframes", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()


## 2. 感知任务和数据标注

除多传感器原始读数外，nuScenes 在**关键帧（2 Hz）**上提供了多种**监督信号**：
- 带类别与属性的 **3D 边界框**：3D 目标检测任务，预测任务和跟踪任务
- **点级语义/全景**标签：2D/3D 语言分割任务
- **矢量地图**（可行驶区域、车道中心线、拓扑连接）：地图理解与预测


下文一律沿用第 1 节中已取得的 **`first_sample`**（当前 `scene` 的首个关键帧；若你修改了随机种子，则对应另一场景的首帧）。运行本节代码前，请确保已执行第 1 节中初始化 `nusc`、`scene`、`first_sample` 的单元格。

**官方任务入口与说明文档**： [3D Detection](https://www.nuscenes.org/object-detection)、[Tracking](https://www.nuscenes.org/tracking)、[Prediction](https://www.nuscenes.org/prediction)、[LiDAR Segmentation](https://www.nuscenes.org/lidarseg)、[Panoptic](https://www.nuscenes.org/panoptic)。

### 2.1 三维目标检测（3D Object Detection）

**预测目标**：对每个关键帧，在统一坐标系下输出一组有向 **3D 边界框**（中心位置、尺寸 $l,w,h$、航向角），并给出 **类别**（完整数据集约 23 类；检测挑战在评测时合并为 **10 类**）、可选 **速度** 与 **属性**（如 `vehicle.moving`、`cycle.with_rider` 等）。

评测主指标为
- **mAP**（多距离阈值下的平均精度）
- **NDS（nuScenes Detection Score）**：综合了 **ATE / ASE / AOE / AVE / AAE** 五项 True Positive 误差，用于刻画框的位置、尺度、朝向、速度与属性质量

运行下方代码展示检测物体投影框；前视相机上的 2D 投影；以及 `LIDAR_TOP` 上的鸟瞰点云与框。

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
from nuscenes.utils.geometry_utils import BoxVisibility

# Multi-sensor figure: six cameras + LiDAR BEV (official explorer layout)
nusc.render_sample(first_sample["token"], box_vis_level=BoxVisibility.ANY, verbose=False)

# Camera-space view: 3D boxes projected to CAM_FRONT (monocular 3D / image-plane QA)
_, ax_cam = plt.subplots(1, 1, figsize=(14, 8))
nusc.render_sample_data(
    first_sample["data"]["CAM_FRONT"],
    with_anns=True,
    box_vis_level=BoxVisibility.ANY,
    ax=ax_cam,
    verbose=False,
)
ax_cam.set_title("CAM_FRONT | projected 3D box wireframes (detection supervision)")
plt.tight_layout()
plt.show()

# LiDAR bird's-eye view with boxes (same targets, LiDAR-centric geometry)
_, ax_bev = plt.subplots(1, 1, figsize=(11, 11))
nusc.render_sample_data(
    first_sample["data"]["LIDAR_TOP"],
    with_anns=True,
    box_vis_level=BoxVisibility.ANY,
    ax=ax_bev,
    verbose=False,
)
ax_bev.set_title("LIDAR_TOP BEV | oriented boxes + point cloud (detection)")
plt.tight_layout()
plt.show()

<mark> 👉 通过 ai 助手的帮助，请绘制出 nuscene-mini 数据集中所有场景出现的物体的类别分布，观察不同类别样本的分布差异</mark>

In [ ]:
# 在这里运行你的代码

<mark> 👉 请查阅网络，搜索 nuscene 数据集上 3D 目标检测的 SOTA 方案是什么？ 你认为，纯视觉方案、点云方法和多传感器融合方案的优缺点是什么？特别是不同属性的物体的检测效果会带来什么差异（例如小目标和大目标）？</mark>

在这里写你的回答，不超过 100 字

### 2.2 LiDAR 语义分割（nuScenes-lidarseg）

**预测目标**：对 **LiDAR 点云中每一个点** 赋予语义类别。

评估指标以 **mIoU** 为主。与检测不同，分割任务通常**不对**远距离或稀有类别做与检测相同的范围裁剪规则。

### 2.3 运动预测（Motion / Trajectory Prediction）

**预测目标**：在给定过去观测与（可选）地图上下文的前提下，对场景中关键交通参与者未来 **若干秒内的平面轨迹** $\{(x_t,y_t)\}$ 进行回归或多模态分布建模。

### 2.4 多目标跟踪（Tracking）与全景分割（Panoptic）

**跟踪**的预测目标是在时间上维护每个交通参与者的 **ID 一致性与 3D 状态**（可看作在检测框上增加跨帧关联）。

评测指标：
- AMOTA / AMOTP
- CLEAR MOT 族指标变体（详见 [Tracking 挑战页](https://www.nuscenes.org/tracking) ）。

全景分割要求在点云上同时提供 **语义 + 实例 ID**（stuff 与 thing 的区分遵循全景分割惯例），并可派生 **全景跟踪**指标：
- **PQ**
- **sPQ**
- **关联误差** 等 （详见 [Panoptic 挑战页](https://www.nuscenes.org/panoptic)）。